In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tf.keras.models import Sequential
from tf.keras.layers import LSTM, Dense

In [ ]:
file_path = "/home/home/code/xucenying/grid-intelligence/notebooks/susanta/consolidated_full.csv"
df=pd.read_csv(file_path)


In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.keys()

In [ ]:
data = df.drop(columns = ['temperature_c', 'humidity_percent', 'cloud_cover_percent',
       'shortwave_radiation_wm2', 'wind_speed_ms', 'temperature_c_forecast', 'humidity_percent_forecast',
       'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast',
       'wind_speed_ms_forecast'])

In [ ]:
data.head(5)

In [ ]:
data.keys()

 # ***Selecting data only from last one year***

In [ ]:
data["datetime_utc"] = pd.to_datetime(data["datetime_utc"], utc=True)

start_date = pd.Timestamp("2025-02-01 00:00:00", tz="UTC")
end_date = pd.Timestamp("2026-02-01 00:00:00", tz="UTC")

data = data[
    (data["datetime_utc"] >= start_date) &
    (data["datetime_utc"] <= end_date)
]

In [ ]:
data.tail()

# Preprocessing and cleaning #


1. Market data → step-like → ffill
2. Weather → smooth physics → interpolation
3. Observed weather → high-trust → preserve with ffill
4. Generation/consumption → continuous signals → interpolate
5. Solar → respects day/night physics
6. Target (price) → never fabricate

In [ ]:
# Ensuring datetime as index
data['datetime_utc'] = pd.to_datetime(data['datetime_utc'])
data = data.set_index('datetime_utc').sort_index()

# -----------------------------
# 1. Defining column groups
# -----------------------------
weather_cols = [
    'temperature_c_observed', 'humidity_percent_observed',
    'cloud_cover_percent_observed',
    'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed'
]

market_cols = [
    'ttf_gas', 'wti_oil', 'brent_oil', 'natural_gas'
]

core_cols = ['generation', 'consumption']
mix_cols = ['generation_renewable', 'generation_non_renewable']

# -----------------------------
# 2. Market variables → forward fill
# -----------------------------
data[market_cols] = data[market_cols].ffill()

# -----------------------------
# 3. Weather variables → time interpolation
# -----------------------------
data[weather_cols] = data[weather_cols].ffill().interpolate(method='time')

# -----------------------------
# 4. Core system variables → interpolate + ffill
# -----------------------------
data[core_cols] = data[core_cols].interpolate().ffill()

# -----------------------------
# 5. Energy mix → interpolate
# -----------------------------
data[mix_cols] = data[mix_cols].interpolate()

# -----------------------------
# 6. Wind generation → interpolate + light smoothing
# -----------------------------
data['wind_onshore'] = (
    data['wind_onshore']
    .interpolate()
    .rolling(3, min_periods=1)
    .mean()
)

# -----------------------------
# 7. Solar radiation (physics-aware)
# -----------------------------
# Nighttime → 0
night_mask = (data.index.hour < 6) | (data.index.hour > 20)
data.loc[night_mask, 'shortwave_radiation_wm2_observed'] = \
    data.loc[night_mask, 'shortwave_radiation_wm2_observed'].fillna(0)

# Daytime → interpolate
data['shortwave_radiation_wm2_observed'] = data['shortwave_radiation_wm2_observed'].interpolate()

# -----------------------------
# 8. Target variable →  NOT IMPUTING ANY VALUES
# -----------------------------
data = data[data['price'].notna()]

# -----------------------------
# 9. Final fallback (rare cases)
# -----------------------------
data = data.fillna(data.mean())

# -----------------------------
# 10. Making data for 15 mins interval
# -----------------------------
data = data.asfreq('15min')

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data.index.to_series().diff().value_counts()

# Feature engineering

In [ ]:
##creating lags
lags = [1,4,8,24,96,192,672,1344]
for lag in lags:
    data[f"price_lag_{lag}"] = data["price"].shift(lag)

In [ ]:
##creating rolling windows by mean:
windows = [1, 4, 8, 24, 96, 192, 672, 1344]

for wm in windows:
    data[f'price_roll_mean_{wm}'] = (
        data['price']
        .shift(1)              # IMPORTANT: avoid leakage
        .rolling(wm)
        .mean()
    )

In [ ]:
##creating rolling windows by standard:
windows = [24, 96, 192, 672, 1344]

for ws in windows:
    data[f'price_roll_std_{ws}'] = (
        data['price']
        .shift(1)
        .rolling(ws)
        .std()
    )

# Creating multi-step target

In [ ]:
horizon = 288

target_cols = []

for h in range(1, horizon + 1):
    col = f'target_t+{h}'
    data[col] = data['price'].shift(-h)
    target_cols.append(col)

data = data.dropna().reset_index(drop=True)

In [ ]:
data.head()

In [ ]:
feature_cols = [
    'generation', 'consumption', 'wind_onshore', 'ttf_gas',
    'generation_renewable', 'generation_non_renewable',
    'wti_oil', 'brent_oil', 'natural_gas',
    'temperature_c_observed', 'humidity_percent_observed',
    'cloud_cover_percent_observed',
    'shortwave_radiation_wm2_observed',
    'wind_speed_ms_observed',

    'price_lag_1', 'price_lag_4', 'price_lag_8',
    'price_lag_24', 'price_lag_96', 'price_lag_192',
    'price_lag_672', 'price_lag_1344',

    'price_roll_mean_24', 'price_roll_mean_96',
    'price_roll_mean_192', 'price_roll_mean_672',

    'price_roll_std_24', 'price_roll_std_96',
    'price_roll_std_192', 'price_roll_std_672'
]

X = data[feature_cols].values
y = data[target_cols].values

In [ ]:
# -------------------------
# 1. Train / Val / Test Split
# -------------------------
n = len(data)
train_end = int(n * 0.7)
val_end   = int(n * 0.9)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:], y[val_end:]

# -------------------------
# 2. Scaling (fit only on train)
# -------------------------
x_scaler = RobustScaler()
y_scaler = RobustScaler()

X_train = x_scaler.fit_transform(X_train)
X_val   = x_scaler.transform(X_val)
X_test  = x_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train)
y_val   = y_scaler.transform(y_val)
y_test  = y_scaler.transform(y_test)

# -------------------------
# 3. Reshape for LSTM
# -------------------------
X_train = X_train[:, None, :]
X_val   = X_val[:, None, :]
X_test  = X_test[:, None, :]

In [ ]:
print(X_train.shape)

In [ ]:
def create_sequences(X, y, window=96, horizon=288):
    Xs, ys = [], []

    for i in range(len(X) - window - horizon):
        Xs.append(X[i:i+window])
        ys.append(y[i+window:i+window+horizon])

    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train, y_train, window=96, horizon=288)

X_val_seq, y_val_seq = create_sequences(X_val, y_val, window=96, horizon=288)

X_test_seq, y_test_seq = create_sequences(X_test, y_test, window=96, horizon=288)


In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(96, X_train_seq.shape[2])),
    LSTM(64),

    Dense(128, activation='relu'),
    Dense(288)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_test_seq, y_test_seq),  # or validation split if you prefer
    epochs=50,
    batch_size=32,
    shuffle=False,   # IMPORTANT for time series
    verbose=1
)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
def inverse_price_multi(pred, y_scaler):
    return y_scaler.inverse_transform(pred)

y_test_real = inverse_price_multi(y_test, y_scaler)
y_pred_real = inverse_price_multi(y_pred, y_scaler)

In [ ]:
# Ensure correct shape (safety)
actual = np.array(y_test_real[0]).flatten()
pred   = np.array(y_pred_real[0]).flatten()

steps = np.arange(1, len(actual) + 1)

plt.figure(figsize=(14,6))

plt.plot(steps, actual, label="Actual (t+1 → t+288)")
plt.plot(steps, pred, label="Predicted (t+1 → t+288)")

plt.title("ENTSO-E Electricity Price Forecast (96 → 288-step)")
plt.xlabel("Future Time Step (hours ahead)")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

plt.show()

# Comments